In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS main.silver;

CREATE OR REPLACE TABLE main.silver.customer_dimension
(
    customer_id INT,
    customer_name STRING,
    city STRING,
    balance DECIMAL(10,2),
    start_date DATE,
    end_date DATE,
    is_current STRING
);

In [0]:
%sql

INSERT INTO main.silver.customer_dimension VALUES
(101,'Rajesh Kumar','Hyderabad',45000,'2026-07-01',NULL,'Y'),
(102,'Priya Sharma','Bengaluru',68000,'2026-07-01',NULL,'Y'),
(103,'Arjun Reddy','Chennai',52000,'2026-07-01',NULL,'Y');

In [0]:
%sql

SELECT *
FROM main.silver.customer_dimension
ORDER BY customer_id;

# Staging daily table

In [0]:
%sql

CREATE OR REPLACE TABLE main.bronze.customer_day2
(
    customer_id INT,
    customer_name STRING,
    city STRING,
    balance DECIMAL(10,2)
);

INSERT INTO main.bronze.customer_day2 VALUES
(101,'Rajesh Kumar','Pune',55000),
(102,'Priya Sharma','Bengaluru',68000),
(103,'Arjun Reddy','Chennai',52000),
(104,'Ananya Gupta','Delhi',78000);

In [0]:
%sql

SELECT *
FROM main.bronze.customer_day2
ORDER BY customer_id;

In [0]:
%sql

UPDATE main.silver.customer_dimension AS target
SET
    end_date = current_date(),
    is_current = 'N'
WHERE
    is_current = 'Y'
AND EXISTS
(
SELECT 1
FROM main.bronze.customer_day2 source
WHERE target.customer_id = source.customer_id
AND
(
target.city <> source.city
OR target.balance <> source.balance
)
);

In [0]:
%sql

SELECT *
FROM main.silver.customer_dimension
ORDER BY customer_id,start_date;

# just check what values we can get 

In [0]:
SELECT
    source.customer_id,
    source.customer_name,
    source.city,
    source.balance,
    current_date(),
    NULL,
    'Y',
    target.customer_id
FROM main.bronze.customer_day2 source

LEFT JOIN main.silver.customer_dimension target
ON source.customer_id = target.customer_id
AND target.is_current='Y'

WHERE target.customer_id IS NULL;

# Insert new version

In [0]:
%sql

INSERT INTO main.silver.customer_dimension
(
    customer_id,
    customer_name,
    city,
    balance,
    start_date,
    end_date,
    is_current
)

SELECT
    source.customer_id,
    source.customer_name,
    source.city,
    source.balance,
    current_date(),
    NULL,
    'Y'
FROM main.bronze.customer_day2 source

LEFT JOIN main.silver.customer_dimension target
ON source.customer_id = target.customer_id
AND target.is_current='Y'

WHERE target.customer_id IS NULL;

In [0]:
%sql

SELECT *
FROM main.silver.customer_dimension
ORDER BY customer_id, start_date;